# 00. Environment Check

**Tier:** Foundations
**Estimated time:** 15 minutes
**Prerequisites:** none
**Source material:** AI Learning curriculum setup (see `.claude/CLAUDE.md`)

## What You'll Learn
- How to verify your Python environment has everything the curriculum needs
- Which compute device (CUDA GPU, Apple MPS, or CPU) your machine will use for the deep-learning notebooks
- How API keys are loaded from a `.env` file — and how to keep running the concept cells even without keys

## Why This Matters
Every later notebook assumes this one passed. Five minutes of checking now saves you from cryptic `ModuleNotFoundError`s and "why is training so slow?" confusion in notebooks 01–11. Think of this as the pre-flight checklist before takeoff.

## 1. Python version

The curriculum targets **Python 3.11+**. The repo itself is managed with [`uv`](https://docs.astral.sh/uv/) and pins 3.13, but anything ≥ 3.11 works. If the check below fails, create the environment with `uv sync` (or `conda env create -f environment.yml`).

In [1]:
import sys, platform

major, minor = sys.version_info[:2]
print(f"Python {platform.python_version()}  ({sys.executable})")
assert (major, minor) >= (3, 11), "Please use Python 3.11 or newer"
print("OK — Python version is fine.")

Python 3.13.7  (/Users/itaizemah-goren/Desktop/ai_learning/.venv/bin/python)
OK — Python version is fine.


## 2. Required packages

These are the libraries the Foundations and Training tiers rely on. The cell imports each one and prints its version, so you can see at a glance what's installed. If anything is missing, install it with `uv add <package>` (or `pip install -r requirements.txt`).

In [2]:
import importlib

# (import name, friendly name) — import name sometimes differs from pip name
packages = [
    ("numpy", "numpy"),
    ("torch", "torch"),
    ("transformers", "transformers"),
    ("tokenizers", "tokenizers"),
    ("sklearn", "scikit-learn"),
    ("matplotlib", "matplotlib"),
    ("sentence_transformers", "sentence-transformers"),
    ("anthropic", "anthropic"),
    ("openai", "openai"),
]

missing = []
for import_name, pip_name in packages:
    try:
        mod = importlib.import_module(import_name)
        version = getattr(mod, "__version__", "?")
        print(f"  {pip_name:24s} {version}")
    except ImportError:
        missing.append(pip_name)
        print(f"  {pip_name:24s} MISSING")

if missing:
    print("\nInstall the missing packages with:")
    print("  uv add " + " ".join(missing))
else:
    print("\nOK — all required packages are importable.")

  numpy                    2.4.6


  torch                    2.10.0


  transformers             5.10.1
  tokenizers               0.22.2


  scikit-learn             1.9.0
  matplotlib               3.10.9


  sentence-transformers    5.5.1


  anthropic                0.105.2


  openai                   2.41.1

OK — all required packages are importable.


## 3. Compute device

PyTorch runs on whatever accelerator it can find. The helper below picks the best available device in this priority order:

1. **CUDA** — an NVIDIA GPU (most cloud machines, gaming PCs)
2. **MPS** — Apple's Metal backend (M-series Macs)
3. **CPU** — always available, just slower

Every deep-learning notebook in this curriculum calls a function like this one, so the same code runs on your laptop and on a GPU box. We keep it tiny and visible — no magic.

In [3]:
import torch

def pick_device():
    """Return the best available torch device, most capable first."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():       # Apple Silicon (M1/M2/M3/M4)
        return torch.device("mps")
    return torch.device("cpu")

device = pick_device()
print("Selected device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
elif device.type == "mps":
    print("Apple Metal (MPS) acceleration is available.")
else:
    print("Running on CPU — fine for the Foundations notebooks, slow for training (07–11).")

# Prove it works: a small tensor op on the chosen device.
x = torch.randn(3, 3, device=device)
print("\nSample tensor on device:\n", x @ x.T)

Selected device: mps
Apple Metal (MPS) acceleration is available.



Sample tensor on device:
 

tensor([[ 0.1516,  0.4722,  0.0196],
        [ 0.4722,  4.4585, -0.0723],
        [ 0.0196, -0.0723,  0.0139]], device='mps:0')


## 4. API keys (optional for Tiers 1–2)

Tiers 1–2 (notebooks 01–11) run **fully offline** — no API key needed. Tier 3+ uses the Anthropic and OpenAI SDKs.

Keys are read from a `.env` file in the repo root (copy `.env.example` → `.env` and fill in your own). We load it with `python-dotenv` if available, otherwise fall back to the normal environment. **This notebook never prints your key** — only whether one is present.

In [4]:
import os

# Load .env if python-dotenv is installed; harmless if it isn't.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

def key_status(name):
    value = os.environ.get(name)
    if value:
        # Show only that it exists and its length — never the key itself.
        return f"set ({len(value)} chars)"
    return "not set"

for key in ["ANTHROPIC_API_KEY", "OPENAI_API_KEY", "HF_TOKEN"]:
    print(f"  {key:20s} {key_status(key)}")

print("\nReminder: notebooks 01–11 do NOT require any of these. "
      "If they're 'not set', you can still complete all of Tiers 1–2.")

  ANTHROPIC_API_KEY    not set
  OPENAI_API_KEY       not set
  HF_TOKEN             not set

Reminder: notebooks 01–11 do NOT require any of these. If they're 'not set', you can still complete all of Tiers 1–2.


## 5. The "no-key" pattern

Notebooks that call a paid API begin with a clearly marked cell like the one below. When no key is present, the concept and visualization cells still run — only the live API call is skipped. You'll see this exact pattern again starting in notebook 13.

In [5]:
HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))

if HAS_ANTHROPIC:
    print("Anthropic key found — live API cells will run in Tier 3+ notebooks.")
else:
    print("No Anthropic key — Tier 3+ live cells will be skipped, explanations still work.")
    print("Set ANTHROPIC_API_KEY in your .env file when you're ready.")

No Anthropic key — Tier 3+ live cells will be skipped, explanations still work.
Set ANTHROPIC_API_KEY in your .env file when you're ready.


## Key Takeaways
- Python **3.11+** plus the packages listed in section 2 are everything Tiers 1–2 need.
- `pick_device()` transparently chooses CUDA → MPS → CPU; the same notebook code runs everywhere.
- API keys live in `.env`, are loaded with `python-dotenv`, and are **never printed** — only their presence is reported.
- The "no-key" guard lets you complete every Foundations/Training notebook offline.

## What's Next
Notebook **01 — Neural Networks** builds a working neural net from scratch in NumPy, so you can see exactly what "the model learns" really means before any transformer machinery appears.